<a href="https://colab.research.google.com/github/fboldt/aulasml/blob/master/aula04c%20-%20cross%20validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔄 Aula 04c: Validação Cruzada e Seleção de Modelos

Nesta aula prática, vamos explorar técnicas robustas de **validação e avaliação de modelos** em Machine Learning.

Entenderemos a diferença entre a divisão simples (Holdout), Validação Cruzada Manual, K-Fold e Repeated K-Fold, usando essas abordagens para encontrar o melhor hiperparâmetro $K$ no algoritmo K-NN.

In [215]:
# Carrega a base Wine e divide em treino e teste
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split

X, y = load_wine(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

### 📉 Avaliação Básica com Holdout (K=1)

#### 🔍 O que este bloco faz?
Treina o K-NN com $K=1$ no conjunto de treino e calcula a acurácia no conjunto de teste.

#### 🎯 Qual a intenção pedagógica?
Estabelecer a linha de base (baseline) inicial. Uma divisão simples em treino e teste (Holdout) pode sofrer com alta variabilidade dependendo do sorteio aleatório inicial, não sendo a forma mais confiável de estimar a performance do modelo em produção.

In [216]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Baseline: Acurácia no teste com K=1
model = KNeighborsClassifier(n_neighbors=1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.6111111111111112


### 📈 Avaliação Básica com Holdout (K=5)

#### 🔍 O que este bloco faz?
Modifica o hiperparâmetro $K$ para 5 vizinhos e reavalia a acurácia no teste.

#### 🎯 Qual a intenção pedagógica?
Observar como a mudança de um hiperparâmetro altera a acurácia. Mas surge um problema: como podemos escolher o melhor $K$ sem olhar diretamente para o conjunto de teste (o que configuraria vazamento de dados)?

In [217]:
# Acurácia no teste com K=5
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.6666666666666666


### 🎯 Divisão de Validação (Holdout com Conjunto de Validação)

#### 🔍 O que este bloco faz?
Subdivide o conjunto de treino original (`X_train`) em novos conjuntos de treino (`X_tr`) e validação (`X_val`). Em seguida, varre diferentes valores de $K$ para encontrar o melhor na validação.

#### 🎯 Qual a intenção pedagógica?
Ensinar o uso correto do **conjunto de validação**. Ele serve como um simulador do conjunto de teste para a busca de hiperparâmetros (model selection), mantendo o conjunto de teste intocado e livre de viés de seleção.

In [227]:
# Criação do conjunto de validação
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2)

best_acc = 0
best_k = 0

for k in range(1, 21, 2):
  model = KNeighborsClassifier(n_neighbors=k)
  model.fit(X_tr, y_tr)
  y_pred = model.predict(X_val)
  acc = accuracy_score(y_val, y_pred)
  print(f"Accuracy for k={k}: {acc}")
  if acc > best_acc:
    best_acc = acc
    best_k = k

print(f"Best accuracy: {best_acc} for k={best_k}")

Accuracy for k=1: 0.6551724137931034
Accuracy for k=3: 0.6551724137931034
Accuracy for k=5: 0.6206896551724138
Accuracy for k=7: 0.5517241379310345
Accuracy for k=9: 0.7586206896551724
Accuracy for k=11: 0.6896551724137931
Accuracy for k=13: 0.6551724137931034
Accuracy for k=15: 0.7241379310344828
Accuracy for k=17: 0.6551724137931034
Accuracy for k=19: 0.6896551724137931
Best accuracy: 0.7586206896551724 for k=9


## 🔄 Validação Cruzada (Cross-Validation)

A validação cruzada divide os dados em $k$ blocos (folds), garantindo que cada bloco seja usado como teste uma vez, enquanto os outros servem para treino. Isso reduz drasticamente a variabilidade da estimativa de acurácia.

### 🛠️ Implementando Validação Cruzada Manual

#### 🔍 O que este bloco faz?
Cria uma função manual `cross_validation` que embaralha os índices dos dados, divide em $K$ folds e executa uma iteração de validação cruzada retornando a acurácia de um fold.

#### 🎯 Qual a intenção pedagógica?
Mostrar o passo a passo lógico e a manipulação de arrays via Numpy para entender como os folds são fatiados e recombinados para treino e validação.

In [274]:
import numpy as np

# Validação Cruzada desenvolvida manualmente
def cross_validation(model, X, y, k=3):
  n = int(len(y)/k)
  idx = np.random.permutation(len(y))
  X = X[idx]
  y = y[idx]
  for i in range(k):
    X_tr = np.concatenate([X[:i*n], X[(i+1)*n:]])
    y_tr = np.concatenate([y[:i*n], y[(i+1)*n:]])
    X_val = X[i*n:(i+1)*n]
    y_val = y[i*n:(i+1)*n]
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_val)
    acc = accuracy_score(y_val, y_pred)
    return acc

acc = cross_validation(model, X_train, y_train)
print(acc)

0.723404255319149


### 🔍 Busca do Melhor K via Validação Cruzada Manual

#### 🔍 O que este bloco faz?
Busca o melhor número de vizinhos $K$ usando nossa função de validação cruzada criada do zero.

#### 🎯 Qual a intenção pedagógica?
Perceber que, devido à natureza aleatória da partição, cada execução dessa validação pode retornar um resultado ligeiramente diferente. Precisamos de mais repetições para obter uma média estável.

In [257]:
# Busca de hiperparâmetro com CV manual
def search_best_k(model, X, y, k=3):
  best_acc = 0
  best_k = 0
  for k in range(1, 21, 2):
    model = KNeighborsClassifier(n_neighbors=k)
    acc = cross_validation(model, X, y)
    print(f"Accuracy for k={k}: {acc}")
    if acc > best_acc:
      best_acc = acc
      best_k = k
  print(f"Best accuracy: {best_acc} for k={best_k}")

search_best_k(model, X_train, y_train)

Accuracy for k=1: 0.723404255319149
Accuracy for k=3: 0.723404255319149
Accuracy for k=5: 0.6595744680851063
Accuracy for k=7: 0.7872340425531915
Accuracy for k=9: 0.574468085106383
Accuracy for k=11: 0.8723404255319149
Accuracy for k=13: 0.6382978723404256
Accuracy for k=15: 0.6170212765957447
Accuracy for k=17: 0.7021276595744681
Accuracy for k=19: 0.7659574468085106
Best accuracy: 0.8723404255319149 for k=11


### 🔄 Validação Cruzada Repetida (Repeated Cross-Validation)

#### 🔍 O que este bloco faz?
Executa a validação cruzada manual múltiplas vezes ($N=10$), retornando a média e o desvio padrão das acurácias obtidas.

#### 🎯 Qual a intenção pedagógica?
A repetição nos permite calcular não apenas o desempenho médio, mas também o desvio padrão (variância), dando-nos uma estimativa estatística de quão confiável e estável é o nosso modelo.

In [281]:
# Validação Cruzada Repetida desenvolvida manualmente
def repeated_cross_validation(model, X, y, k=3, n=10):
  accs = []
  for i in range(n):
    acc = cross_validation(model, X, y)
    accs.append(acc)
  return np.mean(accs), np.std(accs)

mean, std = repeated_cross_validation(model, X_train, y_train)
print(mean, std)

0.7170212765957448 0.05713073013658531


### 🔍 Busca Robusta de K com Validação Cruzada Repetida

#### 🔍 O que este bloco faz?
Busca o melhor $K$ usando a média de 10 repetições da validação cruzada.

#### 🎯 Qual a intenção pedagógica?
Mostrar como essa abordagem estatística estabiliza os valores de acurácia, nos dando muito mais segurança ao selecionar o valor de $K$ ideal.

In [296]:
# Busca de hiperparâmetro com CV Repetida manual
def search_best_k(model, X, y, k=3):
  best_acc = 0
  best_k = 0
  for k in range(1, 21, 2):
    model = KNeighborsClassifier(n_neighbors=k)
    acc, _ = repeated_cross_validation(model, X, y)
    print(f"Accuracy for k={k}: {acc}")
    if acc > best_acc:
      best_acc = acc
      best_k = k
  print(f"Best accuracy: {best_acc} for k={best_k}")

search_best_k(model, X_train, y_train)

Accuracy for k=1: 0.7361702127659574
Accuracy for k=3: 0.7
Accuracy for k=5: 0.6702127659574468
Accuracy for k=7: 0.6659574468085105
Accuracy for k=9: 0.7127659574468084
Accuracy for k=11: 0.6978723404255319
Accuracy for k=13: 0.6914893617021277
Accuracy for k=15: 0.6978723404255319
Accuracy for k=17: 0.7170212765957447
Accuracy for k=19: 0.7255319148936169
Best accuracy: 0.7361702127659574 for k=1


## 🤖 Validação Cruzada com o Scikit-Learn

O Scikit-Learn fornece classes prontas, otimizadas e muito robustas para realizar validação cruzada de forma simplificada.

### 📦 KFold do Scikit-Learn

#### 🔍 O que este bloco faz?
Utiliza a classe `KFold` com embaralhamento ativo para particionar e validar o modelo de forma profissional.

#### 🎯 Qual a intenção pedagógica?
Apresentar a classe oficial `KFold`. Note como o loop iterando sobre `kf.split(X_train)` simplifica o fatiamento dos dados e torna o código limpo e idiomático.

In [305]:
from sklearn.model_selection import KFold

# KFold oficial do Scikit-Learn
kf = KFold(n_splits=3, shuffle=True)
accs = []
for train_index, test_index in kf.split(X_train):
  X_tr, X_val = X_train[train_index], X_train[test_index]
  y_tr, y_val = y_train[train_index], y_train[test_index]
  model = KNeighborsClassifier(n_neighbors=5)
  model.fit(X_tr, y_tr)
  y_pred = model.predict(X_val)
  acc = accuracy_score(y_val, y_pred)
  accs.append(acc)

print(np.mean(accs))

0.7257683215130024


## 🤖 Validação Cruzada com o Scikit-Learn

O Scikit-Learn fornece classes prontas, otimizadas e muito robustas para realizar validação cruzada de forma simplificada.

### 📦 KFold do Scikit-Learn

#### 🔍 O que este bloco faz?
Utiliza a classe `KFold` com embaralhamento ativo para particionar e validar o modelo de forma profissional.

#### 🎯 Qual a intenção pedagógica?
Apresentar a classe oficial `KFold`. Note como o loop iterando sobre `kf.split(X_train)` simplifica o fatiamento dos dados e torna o código limpo e idiomático.

In [314]:
from sklearn.model_selection import KFold

# KFold oficial do Scikit-Learn
kf = KFold(n_splits=3, shuffle=True)
accs = []
for train_index, test_index in kf.split(X_train):
  X_tr, X_val = X_train[train_index], X_train[test_index]
  y_tr, y_val = y_train[train_index], y_train[test_index]
  model = KNeighborsClassifier(n_neighbors=5)
  model.fit(X_tr, y_tr)
  y_pred = model.predict(X_val)
  acc = accuracy_score(y_val, y_pred)
  accs.append(acc)

print(np.mean(accs))

0.6761081560283688


### ⚡ cross_val_score (O jeito mais prático!)

#### 🔍 O que este bloco faz?
Mede a acurácia usando a função utilitária `cross_val_score` integrada com a partição `KFold` configurada.

#### 🎯 Qual a intenção pedagógica?
Apresentar a forma de validação mais rápida e amplamente utilizada do Scikit-Learn. Ela automatiza o loop de treino/validação internamente, retornando um array contendo os scores obtidos em cada fold.

In [318]:
from sklearn.model_selection import cross_val_score

# cross_val_score integrado com KFold
scores = cross_val_score(model, X_train, y_train,
                         scoring='accuracy',
                         cv=KFold(n_splits=3, shuffle=True))
print(scores)
print(np.mean(scores))

[0.66666667 0.76595745 0.68085106]
0.7044917257683215


### ⚡ cross_val_score (O jeito mais prático!)

#### 🔍 O que este bloco faz?
Mede a acurácia usando a função utilitária `cross_val_score` integrada com a partição `KFold` configurada.

#### 🎯 Qual a intenção pedagógica?
Apresentar a forma de validação mais rápida e amplamente utilizada do Scikit-Learn. Ela automatiza o loop de treino/validação internamente, retornando um array contendo os scores obtidos em cada fold.

In [319]:
from sklearn.model_selection import cross_val_score

# cross_val_score integrado com KFold
scores = cross_val_score(model, X_train, y_train,
                         scoring='accuracy',
                         cv=KFold(n_splits=3, shuffle=True))
print(scores)
print(np.mean(scores))

[0.72916667 0.76595745 0.65957447 0.72916667 0.59574468 0.72340426
 0.625      0.78723404 0.78723404 0.75       0.78723404 0.59574468
 0.70833333 0.63829787 0.70212766 0.6875     0.72340426 0.78723404
 0.625      0.70212766 0.68085106 0.6875     0.72340426 0.65957447
 0.77083333 0.65957447 0.72340426 0.58333333 0.74468085 0.70212766]
0.70149231678487


### ⚡ cross_val_score (O jeito mais prático!)

#### 🔍 O que este bloco faz?
Mede a acurácia usando a função utilitária `cross_val_score` integrada com a partição `KFold` configurada.

#### 🎯 Qual a intenção pedagógica?
Apresentar a forma de validação mais rápida e amplamente utilizada do Scikit-Learn. Ela automatiza o loop de treino/validação internamente, retornando um array contendo os scores obtidos em cada fold.

In [320]:
from sklearn.model_selection import cross_val_score

# cross_val_score integrado com KFold
scores = cross_val_score(model, X_train, y_train,
                         scoring='accuracy',
                         cv=KFold(n_splits=3, shuffle=True))
print(scores)
print(np.mean(scores))

Accuracy for k=1: 0.7336436170212767
Accuracy for k=3: 0.6908244680851064
Accuracy for k=5: 0.6808510638297872
Accuracy for k=7: 0.6979757683215129
Accuracy for k=9: 0.6788268321513002
Accuracy for k=11: 0.6998965721040187
Accuracy for k=13: 0.6831560283687943
Accuracy for k=15: 0.703619976359338
Accuracy for k=17: 0.7147901891252957
Accuracy for k=19: 0.7097369976359339
Best accuracy: 0.7336436170212767 for k=1


### 📉 Avaliação Básica com Holdout (K=1)

#### 🔍 O que este bloco faz?
Treina o K-NN com $K=1$ no conjunto de treino e calcula a acurácia no conjunto de teste.

#### 🎯 Qual a intenção pedagógica?
Estabelecer a linha de base (baseline) inicial. Uma divisão simples em treino e teste (Holdout) pode sofrer com alta variabilidade dependendo do sorteio aleatório inicial, não sendo a forma mais confiável de estimar a performance do modelo em produção.

In [321]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Baseline: Acurácia no teste com K=1
model = KNeighborsClassifier(n_neighbors=1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.6111111111111112
